# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is a single object in mlcroissant
print(f"Dataset name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

Let's enumerate all available record sets in the dataset, their `@id`s, and the associated fields:

In [ ]:
# Get all record sets and print their IDs and field IDs
recordsets = getattr(metadata, 'recordSet', [])

if not recordsets:
    print("No record sets listed in top-level metadata. Enumerating them dynamically from the schema...")
    # Use dataset API to fetch all available record set ids
    recordset_ids = dataset.list_record_set_ids()
else:
    # recordsets may be a list of Croissant objects with @id attributes
    recordset_ids = [r['@id'] if isinstance(r, dict) and '@id' in r else r for r in recordsets]
    if not recordset_ids:
        # fallback in case above failed
        recordset_ids = dataset.list_record_set_ids()

print("Available record set @id values:")
for rs_id in recordset_ids:
    print(f"  {rs_id}")

print("\nFields for each record set:")
for rs_id in recordset_ids:
    print(f"\nRecord set '@id': {rs_id}")
    rs = dataset.get_record_set(rs_id)
    fields = getattr(rs, 'field', [])
    field_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else f for f in fields]
    print(f"Fields (@id): {field_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Let's extract all available record sets as dataframes, using their `@id` to ensure reproducibility.

In [ ]:
# Extract data from each record set
record_sets_to_load = recordset_ids  # from previous cell
dataframes = {}

for rs_id in record_sets_to_load:
    try:
        records_iter = dataset.records(record_set=rs_id)
        records = list(records_iter)
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for record set {rs_id}")
        else:
            print(f"No records found for record set {rs_id}")
    except Exception as e:
        print(f"Failed to load records for record set {rs_id}: {e}")

if record_sets_to_load:
    example_record_set = record_sets_to_load[0]
    print(f"\nColumns for record set '{example_record_set}':")
    if example_record_set in dataframes:
        print(dataframes[example_record_set].columns.tolist())
        display(dataframes[example_record_set].head())
    else:
        print("No dataframe was loaded for this record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Note:** Fields and group attributes used below must be referenced by their `@id` as observed in the record set above.

In [ ]:
# EDA for the example record set
from pandas.api.types import is_numeric_dtype
record_set_id = example_record_set  # Use the same as previous section

df = dataframes.get(record_set_id)
if df is not None and not df.empty:
    # List all numeric fields (by @id / column name)
    numeric_fields = [col for col in df.columns if is_numeric_dtype(df[col])]
    print(f"Numeric fields in record set {record_set_id}: {numeric_fields}")
    
    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].quantile(0.80)  # top 20% as threshold example
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        normalized_colname = f"{numeric_field}_normalized"
        filtered_df[normalized_colname] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_colname]].head())

        # Try grouping by first non-numeric field
        group_fields = [col for col in df.columns if col not in numeric_fields]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric fields for grouping.")
    else:
        print("No numeric fields detected in the provided record set.")
else:
    print("No records loaded for this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- Histogram for the selected numeric field
- Bar plot of grouped means by non-numeric field

All field references are by the field `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_fields:
    plt.figure(figsize=(10, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
    
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library via its Croissant schema.

- Data was accessed by referencing all record sets and fields using their unique `@id` attributes, ensuring robust handling.
- We loaded the available record sets as dataframes and identified numeric fields for illustrative EDA and normalization.
- Visualizations such as histograms and grouped bar charts were created to explore the data distributions.

For further analysis, see the field list in Section 2 and adapt the filtering/grouping logic to your research questions!